# Environment Provisioning Pipeline

Creates all catalogs and schemas for an environment directly from `configs/environments/env_settings.json` - the single source of truth:

- **Environment catalog** (`lakehouse` / `lakehouse_test` / ...) with its bronze / silver / gold / monitoring schemas
- **Client catalogs** (e.g. `lakeforge_client_alpha_dev`) via `client_catalog_pattern`, for every client discovered from config subfolders (`configs/<type>/<client>/`)

Safe by default: `dry_run=true` only prints the statements. Set `dry_run=false` to execute. Run once per environment before the first pipeline run there.

In [0]:
# Notebook Parameters
dbutils.widgets.text("config_dir", "", "Config Directory (Optional - Auto-resolved if blank)")
# Stage choices come from the central stage registry (configs/environments/env_settings.json keys)
import json as _json
from pathlib import Path as _Path

def _stage_registry_choices():
    fallback = ["dev", "test", "staging", "prod"]
    bases = []
    try:
        _cfg = dbutils.widgets.get("config_dir").strip()
        if _cfg:
            bases += [_Path(_cfg), _Path(_cfg).parent]
    except Exception:
        pass
    try:
        _nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        _here = _Path("/Workspace" + _nb).parent
        bases += [_here, *_here.parents]
    except Exception:
        pass
    bases += [_Path.cwd(), *_Path.cwd().parents]
    for _base in bases:
        for _reg in (_base / "configs" / "environments" / "env_settings.json",
                     _base / "environments" / "env_settings.json"):
            try:
                if _reg.is_file():
                    _stages = list(_json.loads(_reg.read_text(encoding="utf-8")).keys())
                    if _stages:
                        return _stages
            except Exception:
                continue
    return fallback

_STAGE_CHOICES = _stage_registry_choices()
dbutils.widgets.dropdown("environment", "dev" if "dev" in _STAGE_CHOICES else _STAGE_CHOICES[0], _STAGE_CHOICES, "Environment / Stage")
dbutils.widgets.text("client_id", "", "Client Catalog (blank = env only, 'all' = every client, or a client_id)")
dbutils.widgets.dropdown("dry_run", "true", ["false", "true"], "Dry Run (Audit Only)")

# Resolve config_dir and repo_root generically
import os
import sys
from pathlib import Path

raw_config_dir = ""
try:
    raw_config_dir = dbutils.widgets.get("config_dir").strip()
except Exception:
    raw_config_dir = ""

if raw_config_dir:
    config_dir = raw_config_dir
    repo_root = str(Path(config_dir).parent)
else:
    try:
        # Dynamically discover the notebook's workspace/repo location
        nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        repo_root = str(Path(nb_path).parent.parent)
        config_dir = str(Path(repo_root) / "configs")
    except Exception:
        cwd = Path(os.getcwd())
        if (cwd / "configs").exists():
            repo_root = str(cwd)
            config_dir = str(cwd / "configs")
        elif (cwd.parent / "configs").exists():
            repo_root = str(cwd.parent)
            config_dir = str(cwd.parent / "configs")
        else:
            repo_root = str(cwd)
            config_dir = "configs"

# Workspace file access from cluster compute requires the /Workspace prefix
if not config_dir.startswith("/Workspace") and os.path.isdir("/Workspace" + config_dir):
    config_dir = "/Workspace" + config_dir
if not repo_root.startswith("/Workspace") and os.path.isdir("/Workspace" + repo_root):
    repo_root = "/Workspace" + repo_root

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

environment = dbutils.widgets.get("environment")
client_id = dbutils.widgets.get("client_id").strip() or None
dry_run = dbutils.widgets.get("dry_run").lower() == "true"

print(f"Environment Provisioning: env={environment}, client={client_id or 'n/a'}, dry_run={dry_run}")
print(f"config_dir={config_dir}")

In [0]:
# Provision catalogs & schemas from env_settings.json
import json
from pathlib import Path
from lakeforge.metadata.config_parser import ConfigParser

env_config = ConfigParser.get_environment_config(config_dir, environment)
env_catalog = env_config["catalog"]
schema_keys = ("bronze_schema", "silver_schema", "gold_schema", "monitoring_schema")
schemas = [env_config[k] for k in schema_keys if env_config.get(k)]
client_pattern = env_config.get("client_catalog_pattern")


def discover_clients(cfg_dir):
    """All client ids: config subfolders + 'client_id' fields inside any config JSON."""
    clients = set()
    p = Path(cfg_dir)
    for category in ("ingestion", "dq", "transformations", "governance"):
        cat = p / category
        if cat.is_dir():
            clients |= {d.name for d in cat.iterdir() if d.is_dir()}
    views = p / "views"
    if views.is_dir():
        for layer in ("bronze", "silver", "gold"):
            lyr = views / layer
            if lyr.is_dir():
                clients |= {d.name for d in lyr.iterdir() if d.is_dir()}

    def walk(obj):
        if isinstance(obj, dict):
            for k, v in obj.items():
                if k == "client_id" and isinstance(v, str) and v:
                    clients.add(v)
                walk(v)
        elif isinstance(obj, list):
            for item in obj:
                walk(item)

    for jf in p.rglob("*.json"):
        try:
            with open(jf, "r", encoding="utf-8") as f:
                walk(json.load(f))
        except Exception:
            continue
    return clients


target_clients = []
if client_id == "all":
    target_clients = sorted(discover_clients(config_dir))
elif client_id:
    target_clients = [client_id]

# Build the statements (deduped by catalog name)
statements = []
created = set()


def add_catalog(cat, schema_list):
    if cat in created:
        return
    created.add(cat)
    statements.append(f"CREATE CATALOG IF NOT EXISTS {cat}")
    for s in schema_list:
        statements.append(f"CREATE SCHEMA IF NOT EXISTS {cat}.{s}")


add_catalog(env_catalog, schemas)

# Explicitly declared additional catalogs (env_settings 'additional_catalogs'):
# any extra catalog with its own schemas - omitted 'schemas' inherits the env defaults
for cat, spec in (env_config.get("additional_catalogs") or {}).items():
    add_catalog(cat, (spec or {}).get("schemas") or schemas)

# Client catalogs discovered from configs / requested via the widget
for cl in target_clients:
    if not client_pattern:
        print(f"⚠ no client_catalog_pattern in env_settings for '{environment}' - skipping {cl}")
        continue
    client_cat = client_pattern.format(client_id=cl, environment=environment)
    add_catalog(client_cat, schemas)
    # Landing area for this client's source files: configs/ingestion/<client>/*.json
    # read from /Volumes/<client_catalog>/raw_files/landing/ - provision it with the catalog.
    if f"{client_cat}.raw_files" not in created:
        created.add(f"{client_cat}.raw_files")
        statements.append(f"CREATE SCHEMA IF NOT EXISTS {client_cat}.raw_files")
        statements.append(f"CREATE VOLUME IF NOT EXISTS {client_cat}.raw_files.landing")

print(f"Environment '{environment}': catalog={env_catalog}, schemas={schemas}")
print(f"Client catalogs to provision: {target_clients or '(none)'}")
print(f"dry_run={dry_run} -> {len(statements)} statement(s)\n")

if dry_run:
    for s in statements:
        print("  ", s)
    print("\n(DRY RUN - nothing executed. Set dry_run=false to apply.)")
else:
    ok = fail = 0
    for s in statements:
        try:
            spark.sql(s)
            ok += 1
        except Exception as e:
            fail += 1
            print(f"FAILED: {s}\n  {str(e)[:200]}")
    print(f"✅ Applied {ok} statement(s)" + (f", {fail} failed" if fail else ""))